# DCC Correlation and Investor Sentiment Regression

This notebook merges the monthly DCC correlation dataset with the monthly investor sentiment dataset, then estimates OLS regressions for each Shanghai-market DCC correlation series.

## Model setup

Dependent variables:
- `corr_SH_Hang_Seng`
- `corr_SH_Nikkei`
- `corr_SH_KOSPI`
- `corr_SH_STI`
- `corr_SH_SP500`

Main sentiment specification:

$$DCC_{i,t} = \alpha_i + \beta_i StdISI_t + \epsilon_{i,t}$$

Component specification:

$$DCC_{i,t} = \alpha_i + \beta_1DCEF_t + \beta_2RIPO_t + \beta_3NIPO_t + \beta_4NA_t + \beta_5TURN_t + \beta_6CCI_t + \epsilon_{i,t}$$

Standard errors below use a Newey-West/HAC estimator with an automatic monthly lag length.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

PROJECT_DIR = Path.cwd()
DCC_PATH = PROJECT_DIR / 'dataset' / 'monthly_dcc_correlations' / 'monthly_dcc_correlations.csv'
SENTIMENT_PATH = PROJECT_DIR / 'dataset' / 'outputs' / 'stock_sentiment_dataset_sp500.xlsx'

DCC_PATH, SENTIMENT_PATH

(WindowsPath('C:/Users/20945/Desktop/data/project/dataset/monthly_dcc_correlations/monthly_dcc_correlations.csv'),
 WindowsPath('C:/Users/20945/Desktop/data/project/dataset/outputs/stock_sentiment_dataset_sp500.xlsx'))

In [2]:
dcc = pd.read_csv(DCC_PATH)
sentiment = pd.read_excel(SENTIMENT_PATH)

dcc = dcc.rename(columns={'Month': 'month'}).copy()
dcc['month'] = pd.PeriodIndex(dcc['month'].astype(str), freq='M').astype(str)
sentiment['month'] = pd.PeriodIndex(sentiment['month'].astype(str), freq='M').astype(str)

dcc_cols = [c for c in dcc.columns if c.startswith('corr_')]
sentiment_cols = ['DCEF', 'RIPO', 'NIPO', 'NA', 'TURN', 'CCI', 'ISI', 'StdISI']

data = dcc.merge(sentiment[['month'] + sentiment_cols], on='month', how='inner')
data = data.sort_values('month').reset_index(drop=True)

print(f'DCC rows: {len(dcc):,}; sentiment rows: {len(sentiment):,}; merged rows: {len(data):,}')
print(f'Sample period: {data["month"].min()} to {data["month"].max()}')
display(data.head())

DCC rows: 248; sentiment rows: 248; merged rows: 248
Sample period: 2003-01 to 2023-08


,month,corr_SH_Hang_Seng,corr_SH_Nikkei,corr_SH_KOSPI,corr_SH_STI,corr_SH_SP500,DCEF,RIPO,NIPO,NA,TURN,CCI,ISI,StdISI
0,2003-01,0.473808,0.253952,0.303313,0.369487,0.120369,-0.1034,0.9050,5,3.97,0.1054,97.5,26.44,-1.26
1,2003-02,0.415005,0.214475,0.245187,0.342101,0.087893,-0.1117,0.7691,3,2.85,0.1928,97.7,25.30,-1.05
2,2003-03,0.402510,0.227826,0.210353,0.309029,0.087576,-0.0941,0.6831,6,3.26,0.1328,97.8,26.20,-1.27
3,2003-04,0.269006,0.171265,0.163601,0.184975,0.070369,-0.1399,1.1853,5,5.69,0.1218,97.6,27.69,-1.07
4,2003-05,0.247918,0.119658,0.191135,0.192633,0.113358,-0.1892,1.0623,2,4.21,0.3197,88.7,24.05,-0.70


In [3]:
summary_cols = dcc_cols + sentiment_cols
desc = data[summary_cols].describe().T[['count', 'mean', 'std', 'min', 'max']]
display(desc.round(4))

,count,mean,std,min,max
corr_SH_Hang_Seng,248.0,0.4975,0.0978,0.1889,0.6577
corr_SH_Nikkei,248.0,0.2781,0.0768,0.1076,0.4731
corr_SH_KOSPI,248.0,0.3063,0.0864,0.0781,0.5326
corr_SH_STI,248.0,0.3080,0.0906,0.0574,0.5431
corr_SH_SP500,248.0,0.1236,0.0693,-0.0462,0.3117
DCEF,248.0,-0.1211,0.1301,-0.3321,0.4414
RIPO,248.0,0.6191,0.6737,-0.0521,6.2674
NIPO,248.0,17.3185,15.8620,0.0000,82.0000
NA,248.0,40.2921,41.6088,2.3000,297.4700
TURN,248.0,0.2562,0.1226,0.0852,0.8247


In [4]:
def _normal_p_value(t_stat):
    """Two-sided p-value using the standard normal approximation."""
    if pd.isna(t_stat):
        return np.nan
    return math.erfc(abs(float(t_stat)) / math.sqrt(2))


def _significance_stars(p_value):
    if pd.isna(p_value):
        return ''
    if p_value < 0.01:
        return '***'
    if p_value < 0.05:
        return '**'
    if p_value < 0.10:
        return '*'
    return ''


def _newey_west_covariance(X, residuals, max_lag=None):
    """Newey-West HAC covariance matrix for OLS coefficients."""
    X = np.asarray(X, dtype=float)
    residuals = np.asarray(residuals, dtype=float)
    n, k = X.shape
    if max_lag is None:
        max_lag = int(np.floor(4 * (n / 100) ** (2 / 9)))
    max_lag = max(0, min(int(max_lag), n - 1))

    xu = X * residuals[:, None]
    S = xu.T @ xu
    for lag in range(1, max_lag + 1):
        weight = 1 - lag / (max_lag + 1)
        gamma = xu[lag:].T @ xu[:-lag]
        S += weight * (gamma + gamma.T)

    xtx_inv = np.linalg.inv(X.T @ X)
    return xtx_inv @ S @ xtx_inv


def fit_ols(df, y_col, x_cols, hac_lags=None):
    cols = [y_col] + x_cols
    clean = df[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = clean[y_col].astype(float).to_numpy()
    X_raw = clean[x_cols].astype(float).to_numpy()
    X = np.column_stack([np.ones(len(clean)), X_raw])
    names = ['const'] + x_cols

    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    fitted = X @ beta
    residuals = y - fitted
    n, k = X.shape
    dof = n - k

    sse = float(residuals @ residuals)
    tss = float(((y - y.mean()) @ (y - y.mean())))
    r2 = 1 - sse / tss if tss else np.nan
    adj_r2 = 1 - (1 - r2) * (n - 1) / dof if dof > 0 else np.nan

    cov = _newey_west_covariance(X, residuals, hac_lags)
    se = np.sqrt(np.maximum(np.diag(cov), 0))
    t_stats = beta / se
    p_values = np.array([_normal_p_value(t) for t in t_stats])

    rows = []
    for name, coef, stderr, t_stat, p_value in zip(names, beta, se, t_stats, p_values):
        rows.append({
            'dependent': y_col,
            'term': name,
            'coef': coef,
            'hac_se': stderr,
            't': t_stat,
            'p_value': p_value,
            'stars': _significance_stars(p_value),
            'n_obs': n,
            'r2': r2,
            'adj_r2': adj_r2,
            'hac_lags': int(np.floor(4 * (n / 100) ** (2 / 9))) if hac_lags is None else hac_lags,
        })
    return pd.DataFrame(rows)


def run_regressions(df, y_cols, x_cols, hac_lags=None):
    return pd.concat([fit_ols(df, y, x_cols, hac_lags) for y in y_cols], ignore_index=True)


def coefficient_table(results, terms):
    subset = results[results['term'].isin(terms)].copy()
    subset['coef_with_stars'] = subset['coef'].map(lambda x: f'{x:.4f}') + subset['stars']
    subset['se_text'] = subset['hac_se'].map(lambda x: f'({x:.4f})')
    coef = subset.pivot(index='term', columns='dependent', values='coef_with_stars')
    se = subset.pivot(index='term', columns='dependent', values='se_text')
    ordered_rows = []
    for term in terms:
        if term in coef.index:
            ordered_rows.append(pd.DataFrame([coef.loc[term]], index=[term]))
            ordered_rows.append(pd.DataFrame([se.loc[term]], index=[f'{term} HAC se']))
    return pd.concat(ordered_rows) if ordered_rows else pd.DataFrame()


## Main regression: DCC correlations on standardized sentiment

In [5]:
main_results = run_regressions(data, dcc_cols, ['StdISI'])
main_table = coefficient_table(main_results, ['const', 'StdISI'])

model_stats = (
    main_results[main_results['term'].eq('const')]
    .set_index('dependent')[['n_obs', 'r2', 'adj_r2', 'hac_lags']]
    .T
)

display(main_table)
display(model_stats.round(4))

dependent,corr_SH_Hang_Seng,corr_SH_KOSPI,corr_SH_Nikkei,corr_SH_SP500,corr_SH_STI
const,0.4975***,0.3063***,0.2781***,0.1236***,0.3080***
const HAC se,(0.0126),(0.0108),(0.0094),(0.0080),(0.0113)
StdISI,0.0186,0.0066,0.0096,0.0121**,0.0073
StdISI HAC se,(0.0114),(0.0068),(0.0061),(0.0057),(0.0075)


dependent,corr_SH_Hang_Seng,corr_SH_Nikkei,corr_SH_KOSPI,corr_SH_STI,corr_SH_SP500
n_obs,248.0000,248.0000,248.0000,248.0000,248.0000
r2,0.0778,0.0337,0.0124,0.0139,0.0650
adj_r2,0.0741,0.0298,0.0084,0.0099,0.0612
hac_lags,4.0000,4.0000,4.0000,4.0000,4.0000


## Component regression: DCC correlations on sentiment components

In [6]:
component_vars = ['DCEF', 'RIPO', 'NIPO', 'NA', 'TURN', 'CCI']
component_results = run_regressions(data, dcc_cols, component_vars)
component_table = coefficient_table(component_results, ['const'] + component_vars)

component_stats = (
    component_results[component_results['term'].eq('const')]
    .set_index('dependent')[['n_obs', 'r2', 'adj_r2', 'hac_lags']]
    .T
)

display(component_table)
display(component_stats.round(4))

dependent,corr_SH_Hang_Seng,corr_SH_KOSPI,corr_SH_Nikkei,corr_SH_SP500,corr_SH_STI
const,0.1593*,0.0801,-0.0033,-0.1309*,-0.0162
const HAC se,(0.0917),(0.0818),(0.0735),(0.0699),(0.0832)
DCEF,0.2908***,0.1031*,0.1341***,-0.0132,0.0388
DCEF HAC se,(0.1093),(0.0542),(0.0470),(0.0552),(0.0757)
RIPO,-0.0148,-0.0012,-0.0010,0.0031,0.0009
RIPO HAC se,(0.0118),(0.0109),(0.0109),(0.0090),(0.0126)
NIPO,-0.0001,-0.0001,-0.0009*,0.0000,-0.0011*
NIPO HAC se,(0.0006),(0.0005),(0.0005),(0.0005),(0.0006)
NA,0.0002,0.0002,0.0002,0.0001,0.0002
NA HAC se,(0.0002),(0.0002),(0.0002),(0.0002),(0.0002)


dependent,corr_SH_Hang_Seng,corr_SH_Nikkei,corr_SH_KOSPI,corr_SH_STI,corr_SH_SP500
n_obs,248.0000,248.0000,248.0000,248.0000,248.0000
r2,0.4080,0.2379,0.2157,0.1580,0.1608
adj_r2,0.3933,0.2189,0.1962,0.1371,0.1399
hac_lags,4.0000,4.0000,4.0000,4.0000,4.0000


## Correlation check

In [7]:
corr_check = data[dcc_cols + ['StdISI', 'ISI'] + component_vars].corr()
display(corr_check.loc[dcc_cols, ['StdISI', 'ISI'] + component_vars].round(3))

,StdISI,ISI,DCEF,RIPO,NIPO,NA,TURN,CCI
corr_SH_Hang_Seng,0.279,0.205,0.490,-0.029,0.345,0.123,-0.044,0.503
corr_SH_Nikkei,0.184,0.145,0.243,0.041,0.108,0.097,-0.044,0.441
corr_SH_KOSPI,0.112,0.098,0.258,0.029,0.197,0.040,-0.194,0.394
corr_SH_STI,0.118,0.107,0.054,0.054,-0.007,0.080,-0.006,0.363
corr_SH_SP500,0.255,0.225,0.065,0.126,0.148,0.183,0.109,0.370


## Save regression outputs

In [8]:
OUTPUT_DIR = PROJECT_DIR / 'dataset' / 'outputs'
main_results.to_csv(OUTPUT_DIR / 'dcc_sentiment_regression_stdisi_results.csv', index=False)
component_results.to_csv(OUTPUT_DIR / 'dcc_sentiment_regression_component_results.csv', index=False)
data.to_csv(OUTPUT_DIR / 'dcc_sentiment_merged_monthly.csv', index=False)

print('Saved:')
print(OUTPUT_DIR / 'dcc_sentiment_regression_stdisi_results.csv')
print(OUTPUT_DIR / 'dcc_sentiment_regression_component_results.csv')
print(OUTPUT_DIR / 'dcc_sentiment_merged_monthly.csv')

Saved:
C:\Users\20945\Desktop\data\project\dataset\outputs\dcc_sentiment_regression_stdisi_results.csv
C:\Users\20945\Desktop\data\project\dataset\outputs\dcc_sentiment_regression_component_results.csv
C:\Users\20945\Desktop\data\project\dataset\outputs\dcc_sentiment_merged_monthly.csv
